In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

from config import (
    TAGS_EXCEL_PATH, DATA_CSV_PART1, DATA_CSV_PART2, TARGET_COL,
    FINAL_WEIGHTS, load_tag_lists,
)

# Итоговый файл

- `auto_lover_agg_coef` [0, 1]
- `auto_lover_agg_coef_bin` (0/1)
- `shopping_agg_coef` [0, 1]
- `shopping_agg_coef_bin` (0/1)
- `alcohol_agg_coef` [0, 1]
- `alcohol_agg_coef_bin` (0/1)
- `agg_alcohol` (0–20)

In [ ]:
tags_descriptions = pd.read_excel(TAGS_EXCEL_PATH, sheet_name='HT_list')
tag_lists = load_tag_lists(tags_descriptions)

part1 = pd.read_csv(DATA_CSV_PART1, encoding='cp1251', delimiter=',')
part2 = pd.read_csv(DATA_CSV_PART2, encoding='cp1251', delimiter=',')
casco_hashtags_full = pd.merge(part1, part2, on='POLICY_ZV', how='inner')
casco_hashtags_full[TARGET_COL] = casco_hashtags_full['CLAIMS_PART_DAM_COUNT'].astype(bool).astype(int)
casco_hashtags_full.set_index('POLICY_ZV', inplace=True)

casco_hashtags_full.drop(columns=['TAG_JOIN_IND'], inplace=True)
casco_hashtags_full['SUM'] = casco_hashtags_full.filter(like='TAG_').fillna(0).sum(axis=1)
casco_hashtags_full = casco_hashtags_full[casco_hashtags_full['SUM'] > 0]
print('shape:', casco_hashtags_full.shape)

## Расчёт 3 признаков

In [ ]:
BINARY_THRESHOLD = 0.5
N_QUANTILES = 20

ARTIFACTS_DIR = Path('artifacts')
CSV_DIR = ARTIFACTS_DIR / 'csv'
for p in (CSV_DIR,):
    p.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = CSV_DIR / 'all_groups_agg_coef.csv'
result = pd.DataFrame(index=casco_hashtags_full.index)

# auto_lover и shopping — взвешенная сумма по FINAL_WEIGHTS + MinMax + binary
for group_name, tag_list_key in [('auto_lover', 'auto_lover_list'), ('shopping', 'shopping_features_list')]:
    tags = tag_lists[tag_list_key]
    weights = FINAL_WEIGHTS[group_name]
    w = pd.Series({t: float(weights.get(t, 0.0)) for t in tags}, dtype=float)

    X = casco_hashtags_full.reindex(columns=tags, fill_value=0).fillna(0).astype(float)
    score = X.mul(w, axis=1).sum(axis=1)
    score_norm = MinMaxScaler().fit_transform(score.values.reshape(-1, 1)).flatten()

    result[f'{group_name}_agg_coef'] = score_norm
    result[f'{group_name}_agg_coef_bin'] = (score_norm >= BINARY_THRESHOLD).astype(int)

# alcohol — простая сумма TAG + MinMax + binary + квантильный биннинг 0–20
alcohol_tags = tag_lists['alcohol_features_list']
X_alc = casco_hashtags_full.reindex(columns=alcohol_tags, fill_value=0).fillna(0).astype(float)
score_alc = X_alc.sum(axis=1)
score_alc_norm = MinMaxScaler().fit_transform(score_alc.values.reshape(-1, 1)).flatten()
result['alcohol_agg_coef'] = score_alc_norm
result['alcohol_agg_coef_bin'] = (score_alc_norm >= BINARY_THRESHOLD).astype(int)

quantiles_alc = np.linspace(0, 1, N_QUANTILES + 2)
bins_alc = np.unique(np.quantile(score_alc_norm, quantiles_alc))
result['agg_alcohol'] = np.clip(
    np.digitize(score_alc_norm, bins_alc) - 1, 0, N_QUANTILES
)

result.to_csv(OUTPUT_PATH, index=True, encoding='utf-8-sig')
print(f'Сохранено: {OUTPUT_PATH}')
print(f'Колонки: {list(result.columns)}')
print(result.describe())

## Анализ claim rate

In [ ]:
coef_cols = [c for c in result.columns if not c.endswith('_bin')]
fig, axes = plt.subplots(1, len(coef_cols), figsize=(6 * len(coef_cols), 5))

for ax, col in zip(axes, coef_cols):
    bins = pd.qcut(result[col].rank(method='first'), q=10, labels=False)
    analysis = (
        pd.DataFrame({'bin': bins, 'target': casco_hashtags_full[TARGET_COL]})
        .groupby('bin')['target']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'claim_rate', 'count': 'objects'})
    )
    ax.bar(analysis.index.astype(str), analysis['claim_rate'], color='steelblue')
    ax.set_title(f'{col}: claim_rate по децилям')
    ax.set_xlabel('дециль')
    ax.set_ylabel('claim_rate')

plt.tight_layout()
plt.show()